In [1]:
print("hello world")

hello world


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

text = "The food is a"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

print("Input text:")
print(text)

print("\nToken IDs:")
print(inputs["input_ids"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 575.39it/s]

Input text:
The food is a

Token IDs:
tensor([[ 464, 2057,  318,  257]])

Tokens:
['The', 'Ġfood', 'Ġis', 'Ġa']


In [6]:
with torch.no_grad():
    outputs = model(**inputs)

# --------------------------------------------------
# 5. Get logits
# --------------------------------------------------
logits = outputs.logits

print("\nLogits shape:")
print(logits.shape)

# --------------------------------------------------
# 6. Get logits of LAST token
# --------------------------------------------------
last_token_logits = logits[:, -1, :]

print("\nLast-token logits shape:")
print(last_token_logits.shape)

# --------------------------------------------------
# 7. Find most likely next token
# --------------------------------------------------
next_token_id = torch.argmax(
    last_token_logits,
    dim=-1
)

print("\nPredicted next token ID:")
print(next_token_id.item())

print("\nPredicted next token:")
print(tokenizer.decode(next_token_id))


Logits shape:
torch.Size([1, 4, 50257])

Last-token logits shape:
torch.Size([1, 50257])

Predicted next token ID:
845

Predicted next token:
 very


In [7]:
import random
import numpy as np
import torch

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

#Do matrix multiplications faster, without breaking training quality.
# This controls how aggressive PyTorch is when using fast math.
# Faster & stable matmul on NVIDIA GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

max_seq_length = 4096        # Demonstrates long-context + RoPE scaling
dtype = None                # Auto-detect (FP16 on T4, BF16 on A100/L4)
load_in_4bit = True         # Memory-efficient QLoRA-style loading

In [8]:
assert torch.cuda.is_available()

In [15]:
import unsloth  # Must be first!
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig


c:\Users\SiddheshLihe\Transformer-LLM-Training-Tuning-Reasoning\venv\Lib\site-packages\unsloth\__init__.py:1606: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'self' is not defined

In [9]:
#We load a pre-quantized 4-bit TinyLlama model provided by Unsloth.
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=max_seq_length,  # Enables long-context via RoPE scaling
    dtype=dtype,                    # Auto FP16 / BF16
    load_in_4bit=load_in_4bit,      # QLoRA-style memory efficiency
)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\SiddheshLihe\Transformer-LLM-Training-Tuning-Reasoning\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3823, in run_code
  File "C:\Users\SiddheshLihe\AppData\Local\Temp\ipykernel_16448\3536637987.py", line 4, in <module>
    model, tokenizer = FastLanguageModel.from_pretrained(
                       ^^^^^^^^^^^^^^^^^
NameError: name 'FastLanguageModel' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\SiddheshLihe\Transformer-LLM-Training-Tuning-Reasoning\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2279, in showtraceback
  File "c:\Users\SiddheshLihe\Transformer-LLM-Training-Tuning-Reasoning\venv\Lib\site-packages\IPython\core\ultratb.py", line 1300, in structured_traceback
  File "c:\Users\SiddheshLihe\Transformer-LLM-Training-Tuning-Reasoning\venv\Lib\site-packages\IPython\core\ultratb.py", line 1166, in structu